In [ ]:
import numpy as np
import jax.numpy as jnp
from hair_module import (
    interpolate_emmisivity_vmap,
    detect_noisy_bands,
    destripe,
    denoise,
    correct_band,
    lookup_B_vector_batch,
)

from hadar_module_v2 import (
    load_data,
    process_sky_spectrum,
    generate_planck_lut,
    plot_T_map,
    plot_e_map,
    plot_bright_map,
    plot_hsi,
    hadar_solver_v2,
    get_config,
)


def main_stage_1(
    file_path,
    file_name,
    row_mean_threshold,
    band_problem_ratio,
    max_bad_band_ratio,
    lambda_TVx=0.025,
    lambda_TVy=0.12,
    lambda_HOTV=0.1,
):

    hsi_noisy, hsi_wav, wav_unit = load_data(
        filepath=file_path,
        hsi_file_name=file_name,
        value2radiance=0.01,
        hsi_extension_name="bsq",
    )

    noisy_band_indices, stripe_strengths = detect_noisy_bands(
        hsi_noisy,
        row_mean_threshold,
        band_problem_ratio,
        max_bad_band_ratio,
    )

    all_band_indices = np.arange(len(hsi_wav))
    good_band_indices = np.setdiff1d(all_band_indices, noisy_band_indices).astype(int)

    _, hsi_destriped = destripe(
        jnp.asarray(stripe_strengths),
        jnp.asarray(hsi_noisy),
        lambda_TVx=lambda_TVx,
        lambda_TVy=lambda_TVy,
        lambda_HOTV=lambda_HOTV,
    )

    hsi_denoised = denoise(hsi_destriped)

    return (
        hsi_wav,
        wav_unit,
        hsi_noisy,
        hsi_denoised,
        good_band_indices,
        all_band_indices,
    )


def main_stage_2(
    working_wav, hsi_wav, wav_unit, good_band_indices, hsi_denoised, simulated_sky
):
    calibrated_sky = simulated_sky * np.max(hsi_denoised) * 50

    solver_config = get_config()
    e_opt, T_opt, bright_opt, hsi_recovered = hadar_solver_v2(
        working_wav,
        hsi_wav,
        calibrated_sky,
        hsi_denoised,
        good_band_indices,
        solver_config,
    )

    # Due to privilege restrictions, Code about HADAR is not provided in this release.

    return e_opt, bright_opt, T_opt, hsi_recovered

In [ ]:
file_input_folder = "./data"
file_name = "Input file name here"

(
    hsi_wav,
    wav_unit,
    hsi_noisy,
    hsi_denoised,
    good_band_indices,
    all_band_indices,
) = main_stage_1(
    file_input_folder,
    file_name,
    row_mean_threshold=1.005,
    band_problem_ratio=0.1,
    max_bad_band_ratio=0.7,
    lambda_TVx=0.5,
    lambda_TVy=0.5,
    lambda_HOTV=0.025,
)

fig_sky_denoised, point = plot_hsi(hsi_denoised, hsi_wav, band_idx=127, base_height=150)
display(fig_sky_denoised)

In [ ]:
sky, base = process_sky_spectrum(
    np.asarray(hsi_denoised),
    None,
    [69, 444],
    "ALS",
    lam=0.1e1,
    p=0.00001,
    niter=50,
    last_IMF=5,
)

In [ ]:
working_wav, simulated_sky, srf_fwhm = correct_band(
    sky,
    hsi_wav,
    "./highres_ref_sky.txt",
)
print(f"hsi_wav.min(), hsi_wav.max() = {hsi_wav.min()}, {hsi_wav.max()}")
print(
    f"working_wav.min(), working_wav.max() = {working_wav.min()}, {working_wav.max()}"
)
print(srf_fwhm)

In [ ]:
e_opt, bright_opt, T_opt, hsi_recovered = main_stage_2(
    working_wav,
    hsi_wav,
    wav_unit,
    good_band_indices,
    hsi_denoised,
    simulated_sky,
)

In [ ]:
fig_T_opt = plot_T_map(T_opt, base_height=400)
display(fig_T_opt)

fig_e_opt = plot_e_map(e_opt, hsi_wav, band_idx=110, base_height=300)
display(fig_e_opt)

fig_bright_opt = plot_bright_map(bright_opt, brightmap="gray", base_height=300)
display(fig_bright_opt)